In [2]:
import pandas as pd 
import os
import pymysql
import xml.etree.ElementTree as ET

In [5]:
# This script is used to insert the file name and file path of genomic data into the database.
# (db = tgca_gbm, table = raw_file_name_genomic_data)
conn = pymysql.connect(host='localhost', user='mmc', password='root', db='tcga_gbm')

def insert_file_records(conn, table_name, file_records):
    """
    Insert a list of (file_name, file_path, root_file_name) tuples into the specified table.
    """
    with conn.cursor() as cursor:
        sql = f"INSERT INTO {table_name} (file_name, file_path, root_file_name) VALUES (%s, %s, %s)"
        cursor.executemany(sql, file_records)
    conn.commit()

def collect_file_records(base_path, extension):
    records = []
    for root, dirs, files in os.walk(base_path):
        root_id = os.path.basename(root.rstrip(os.sep))
        for file in files:
            if file.endswith(extension):
                records.append((file, os.path.join(root, file), root_id))
    return records



In [6]:
# The insertion logic is handled by the insert_file_records function in the previous cell.
# Insert genomic data records
truncate_qry = "truncate table raw_file_name_genomic_data;"
conn.cursor().execute(truncate_qry)
genomic_data_path = "/home/licongcong/Desktop/TCGA/tcga-gbm/genomic_data"
genomic_records = collect_file_records(genomic_data_path, ".tsv")
insert_file_records(conn, "raw_file_name_genomic_data", genomic_records)



In [7]:
# Insert WSI file records
truncate_qry = "truncate table raw_file_name_WSI;"
conn.cursor().execute(truncate_qry)
wsi_data_path = "/home/licongcong/Desktop/TCGA/tcga-gbm/WSI_image"
wsi_records = collect_file_records(wsi_data_path, ".svs")
insert_file_records(conn, "raw_file_name_WSI", wsi_records)



In [8]:
# Insert clinic file records
truncate_qry = "truncate table raw_file_name_clinic;"
conn.cursor().execute(truncate_qry)
clinic_data_path = "/home/licongcong/Desktop/TCGA/tcga-gbm/clinical"
clinic_records = collect_file_records(clinic_data_path, ".xml")
insert_file_records(conn, "raw_file_name_clinic", clinic_records)

In [9]:
def extract_xml_data(file_path):
    """
    Extract patient_id and bcr_patient_uuid from a clinical XML file.
    """
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        
        # Function to find element by local name, ignoring namespace
        def find_by_local_name(element, local_name):
            for elem in element.iter():
                if elem.tag.split('}')[-1] == local_name:  # Extract local name from {namespace}localname
                    return elem
            return None
        
        # Extract patient_id and bcr_patient_uuid using flexible search
        patient_id_elem = find_by_local_name(root, 'bcr_patient_barcode')
        bcr_patient_uuid_elem = find_by_local_name(root, 'bcr_patient_uuid')
        age_patient = find_by_local_name(root, 'age_at_initial_pathologic_diagnosis')
        gender = find_by_local_name(root, 'gender')
        vital_status = find_by_local_name(root, 'vital_status')
        therapy_type = find_by_local_name(root, 'therapy_type')

        
        result = {
            'patient_id': patient_id_elem.text if patient_id_elem is not None else None,
            'bcr_patient_uuid': bcr_patient_uuid_elem.text if bcr_patient_uuid_elem is not None else None,
            'age_patient': age_patient.text if age_patient is not None else None,
            'gender': gender.text if gender is not None else None,
            'vital_status': vital_status.text if vital_status is not None else None,
            'therapy_type': therapy_type.text if therapy_type is not None else None
        }
        
        return result
    except Exception as e:
        print(f"Error parsing {file_path}: {e}")
        import traceback
        traceback.print_exc()
        return {'patient_id': None, 'bcr_patient_uuid': None}
    
def insert_xml_data(conn, table_name, xml_records):
    """
    Insert extracted XML data into the specified table.
    """
    with conn.cursor() as cursor:
        sql = f"INSERT INTO {table_name} (patient_id, bcr_patient_uuid, age, gender, vital_status, therapy_type) VALUES (%s, %s, %s, %s, %s, %s)"
        cursor.executemany(sql, [(record['patient_id'], record['bcr_patient_uuid'], record['age_patient'], record['gender'], record['vital_status'], record['therapy_type']) for record in xml_records])
    conn.commit()


In [10]:
clinic_data_path = "/home/licongcong/Desktop/TCGA/tcga-gbm/clinical"
for root, dirs, files in os.walk(clinic_data_path):
    for file in files:
        if file.endswith(".xml"):
            xml_data = extract_xml_data(os.path.join(root, file))
            print(xml_data)
            insert_xml_data(conn, "raw_clinic_data", [xml_data])

{'patient_id': 'TCGA-28-5219', 'bcr_patient_uuid': '567a8bf1-3793-46bc-9943-16302df056ce', 'age_patient': '47', 'gender': 'FEMALE', 'vital_status': 'Alive', 'therapy_type': 'Chemotherapy'}
{'patient_id': 'TCGA-14-1829', 'bcr_patient_uuid': '47a4161c-9c61-48f5-b9bc-a6d1acad4e5a', 'age_patient': '57', 'gender': 'MALE', 'vital_status': 'Alive', 'therapy_type': 'Hormone Therapy'}
{'patient_id': 'TCGA-08-0353', 'bcr_patient_uuid': '872cb8e8-cd79-4c0b-9b54-5553b07a6ca6', 'age_patient': '58', 'gender': 'MALE', 'vital_status': 'Dead', 'therapy_type': 'Chemotherapy'}
{'patient_id': 'TCGA-19-2619', 'bcr_patient_uuid': 'f2de4316-026c-4c2d-b091-47bc0ec8612a', 'age_patient': '55', 'gender': 'FEMALE', 'vital_status': 'Alive', 'therapy_type': 'Chemotherapy'}
{'patient_id': 'TCGA-12-1088', 'bcr_patient_uuid': 'ac72738c-8b33-4f03-917c-86b114a2b9bd', 'age_patient': '53', 'gender': 'FEMALE', 'vital_status': 'Dead', 'therapy_type': 'Chemotherapy'}
{'patient_id': 'TCGA-76-6191', 'bcr_patient_uuid': '8c3b28

In [11]:
print("Insertion completed by the cell above.")
conn.close()
print("Done!")

Insertion completed by the cell above.
Done!
